# Building an AI Tutor with Gradio (WebApp)

In [16]:
from __future__ import annotations

import gradio as gr
import os

from openai import OpenAI
from IPython.display import display, Markdown
from dotenv import load_dotenv

In [5]:
# Loading OpenAI Client
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

print("Client Loaded Successfully")

Client Loaded Successfully


In [6]:
# Display Markdown Text
def display_markdown(text):
    display(Markdown(text))

## Building the AI Tutor Function

In [29]:
def get_ai_tutor_response(user_question):
    """Gets the user question and sends it to OpenAI Client to answer the question

    Args:
        user_question (str): The question to ask the user

    Returns:
        Response: An OpenAI Response object
    """
    system_prompt = "You are a helpful and patient tutor. Your job is to answer questions provided to you with clarity while also including real world examples that the user can relate to while providing real world examples and analogies."

    try:
        stream = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            temperature=0.7,
            stream=True
        )

        full_result = ""

        for chunks in stream:
            if chunks.choices[0].delta and chunks.choices[0].delta.content:
                partial_result = chunks.choices[0].delta.content
                full_result += partial_result
                yield full_result

    except Exception as e:
        print(f"Error: {e}")

## General test for API Calling

In [9]:
test_question = "Explain whats a Carburetor in Vehicles"
result = get_ai_tutor_response(test_question)

In [11]:
display_markdown(result)

A carburetor is a device in internal combustion engines that mixes air with a fine spray of liquid fuel, typically gasoline, to create a combustible mixture that can be ignited in the engine's cylinders. While many modern vehicles use fuel injection systems instead, carburetors were widely used in older cars and some motorcycles.

### How It Works:

1. **Air Intake:** When you press the accelerator pedal, air is drawn into the engine. The carburetor has an air intake that allows this air to flow in.

2. **Venturi Effect:** Inside the carburetor, there is a narrowed section called the venturi. As air passes through this narrower area, it speeds up, creating a drop in pressure. This drop in pressure helps to draw fuel from a reservoir (the float chamber) into the airstream.

3. **Mixing:** The fuel mixes with the incoming air to form a vaporized mixture. The ratio of air to fuel is critical for efficient combustion. If there's too much fuel (rich mixture), the engine may run poorly, and if there's too little (lean mixture), it might not run at all.

4. **Delivery to the Engine:** The air-fuel mixture is then delivered to the engine's cylinders, where it is compressed and ignited by the spark plugs.

### Real-World Examples:

- **Classic Cars:** Many classic cars, like the Ford Mustang from the 1960s, used carburetors. Enthusiasts often appreciate carburetors for their simplicity and mechanical charm, as well as the ability to tune them manually for performance.

- **Motorcycles:** Some older motorcycles still use carburetors. For example, many models from Honda, Yamaha, and Kawasaki from the 1980s and 1990s have carbureted engines.

- **Lawn Equipment:** Carburetors are also found in small engines, such as those in lawnmowers and chainsaws. This equipment often uses simpler carburetors than those found in cars, making them easier to maintain.

### Transition to Fuel Injection:

While carburetors were common in vehicles for much of the 20th century, fuel injection systems began to take over in the 1980s and 1990s. Fuel injection offers better fuel efficiency, more precise fuel delivery, and lower emissions. In modern vehicles, you’ll typically find electronic fuel injection systems that automatically adjust the air-fuel mixture based on various factors, such as engine temperature and load.

In summary, while carburetors are less common in today’s vehicles, they represent an important step in the evolution of engine technology, and understanding them provides insight into how combustion engines work.

## Test with prompt modified to be UNHELPFUL and IMPATIENT

In [13]:
test_question = "Explain whats a Carburetor in Vehicles"
result = get_ai_tutor_response(test_question)

In [14]:
display_markdown(result)

A carburetor is a device in older vehicles that mixes air and fuel for the engine. It’s like the chef in a kitchen who decides how much of each ingredient to add to a dish. When you accelerate, it's supposed to provide the right amount of fuel and air to keep the engine running smoothly. 

For example, think about how you might mix a drink. If you add too much syrup, it's overly sweet; too little, and it's too bland. A carburetor tries to find that balance for an engine. But honestly, most modern cars use fuel injection instead because it’s more efficient and reliable. So, unless you're working on a classic car or some kind of vintage project, why do you even care about carburetors? You’d be better off learning about fuel injection systems.

## Creating GRADIO Interactive Interface

In [31]:
ai_tutor_interface = gr.Interface(
    fn=get_ai_tutor_response,
    inputs=gr.Textbox(placeholder="Ask me something?", label="Quest Away:", lines=2),
    outputs=gr.Markdown(label="AI Tutor Response(Streaming...)", container=True, height=250),
    title="🤖 AI Tutor 📖",
    description="Enter your questions below to learn something new today",
    flagging_mode="never"
)

ai_tutor_interface.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
